# NB1 · Problemin tanımlanması

**Üretken Yapay Zekâ Araçları ile Klinik Karar Destek Sistemleri Geliştirilmesi**  
Sağlık Bilimlerinde Teknoloji ve Yapay Zekâ Okuryazarlığı Eğitimi · Akdeniz Üniversitesi · 18 Eylül 2026

Prof. Dr. Utku Köse · Süleyman Demirel Üniversitesi, Bilgisayar Mühendisliği Bölümü  
Yapay Zekâ Uygulama ve Araştırma Merkezi (YAZEM) Müdürü · utkukose@sdu.edu.tr

---

Bu defterde kod üretimi asgari düzeydedir. Amaç, kuracağınız sistemi yedi soruyla
tanımlamak ve bu tanımı sonraki bütün istemlerin girdisi hâline getirmektir.

Atölyenin geri kalanında her istem bu kartı içerecektir. Kart ne kadar belirsizse
üretilen kod da o kadar belirsiz olur.


## Hazırlık


In [ ]:
!pip -q install pandas numpy scikit-learn matplotlib

import urllib.request

REPO = 'https://raw.githubusercontent.com/utkukose/cdss-genai-NB-lecture/main'
for modul in ['checks.py', 'evaluate.py', 'explain.py', 'safety.py',
              'mimic_web.py', 'pipeline.py']:
    urllib.request.urlretrieve(f'{REPO}/workshop/{modul}', modul)

import numpy as np
import pandas as pd
import checks, evaluate as ev, explain as ex, safety as sf

checks.LANG = ev.LANG = sf.LANG = 'tr'

print('Hazır.')


---

## Adım 1 · Kartın doldurulması

Aşağıdaki hücrede yedi soru bulunmaktadır. Hücre ortak senaryonun cevaplarıyla
doldurulmuştur; kendi probleminiz için bu metinleri değiştiriniz.

Bir soruyu cevaplayamıyorsanız boş bırakmak yerine bilmediğinizi ve gerekçesini
yazınız. Bilinmeyen bir prevalans, yazılmamış bir prevalanstan daha kullanışlıdır;
ilki isteme aktarılır, ikincisi sessizce varsayıma dönüşür.


In [ ]:
KART = {
    'karar': (
        'Yoğun bakıma kabul edilen hastanın üç günden uzun kalıp kalmayacağının '
        'öngörülmesi. Sistem kabulden altı saat sonra çıktı üretir ve yatak '
        'kapasitesi planlamasına girdi sağlar.'
    ),
    'kullanici': (
        'Yoğun bakım sorumlu hekimi ve yatak yönetiminden sorumlu hemşire. Sabah '
        'viziti sırasında, hasta listesi gözden geçirilirken kullanılır.'
    ),
    'veri_turu': (
        'Rutin hastane verisi. Demografik bilgi, yatış bağlamı ve ilk altı saate '
        'ait vital bulgular ile laboratuvar sonuçları.'
    ),
    'sonuc': (
        'Yoğun bakım yatış süresinin üç günü aşması. Yatış ve çıkış zamanlarından '
        'hesaplanır ve rutin kayıtlarda doğrudan bulunur.'
    ),
    'prevalans': (
        'Bu kohortta yaklaşık üçte bir. Kendi biriminizdeki oran farklı olacaktır; '
        'bilmiyorsanız bunu da yazınız.'
    ),
    'maliyet': (
        'Kaçırma daha pahalıdır. Uzun kalacak hastanın öngörülmemesi kapasitenin '
        'plansız dolmasına ve yeni hasta kabulünün gecikmesine yol açar. Yanlış '
        'alarmın maliyeti gereksiz bir planlama toplantısıdır.'
    ),
    'zarar': (
        'Yanlış negatifte kapasite planı yetersiz kalır ve devir kararı gecikir. '
        'Devretme kuralı: Sistem çekimser kaldığında veya hasta eğitim '
        'popülasyonuna benzemediğinde karar sorumlu hekime geçer.'
    ),
}


### Kontrol 1


In [ ]:
checks.check_canvas(KART)


---

## Adım 2 · Kartın belirtime dönüştürülmesi

Kart insan diliyle yazılmıştır. Sonraki defterlerde kullanılabilmesi için yapılandırılmış
hâle getirilmesi gerekir.

Bu adımda üretken yapay zekâ aracından karta dayalı bir belirtim üretmesi istenecektir.
İstemin en önemli kısmı son paragrafıdır. Araca, eksik veya çelişkili gördüğü yerleri
kendiliğinden doldurmaması söylenmektedir. Bu talimat verilmediğinde araç her durumda
eksiksiz görünen bir belirtim üretir ve doldurduğu boşlukları belirtmez.


### İstem 1

Aşağıdaki hücreyi çalıştırınız, çıkan metni kopyalayıp yapay zekâ aracına veriniz.


In [ ]:
ISTEM = '''Bir klinik karar destek sistemi tasarlıyorum. Problem tanımım şu:

{kart}

Bu tanıma dayanarak bir belirtim üret ve sonucu Python sözlüğü olarak ver.

KABUL ÖLÇÜTLERİ
spec adında bir Python sözlüğü üret. Anahtarları tam olarak şunlar olsun:
  karar_ani        -> sistemin çıktı ürettiği an, tek cümle
  hedef_tanimi     -> sonucun kesin tanımı ve zaman penceresi
  mevcut_girdiler  -> karar anında erişilebilir değişkenlerin listesi
  yasak_girdiler   -> ancak sonradan erişilebilecek değişkenlerin listesi
  oncelik          -> duyarlılık veya özgüllük, hangisi öncelenmeli
  oncelik_gerekce  -> bu seçimin gerekçesi, tek paragraf
  zarar_senaryolari-> sistemin hastaya zarar verebileceği üç somut yol, liste
  kapsam_disi      -> sistemin açıkça yapmadığı işler, liste
  acik_sorular     -> cevaplarımdaki eksik veya çelişkili noktalar, liste

Kodu yalnızca sözlüğü oluşturacak biçimde yaz, başka çıktı üretme.
Eksik veya kendi içinde tutarsız bulduğun noktaları kendin doldurma; bunları
acik_sorular listesine yaz. Bu liste boş kalmamalıdır.'''

print(ISTEM.format(kart='\n\n'.join(f'{k}: {v}' for k, v in KART.items())))


In [ ]:
# Ürettiğiniz kodu bu hücreye yapıştırınız ve çalıştırınız.


### Kontrol 2


In [ ]:
checks.check_canvas(spec, min_chars=10)


---

## Belirtimin okunması

Üretilen belirtimde önce `acik_sorular` listesini okuyunuz. Bu liste kartınızın zayıf
noktalarını gösterir ve aynı noktalar kuracağınız sistemin de zayıf noktaları olacaktır.
Liste boş geldiyse şüpheleniniz; yedi cümlelik bir tanımın hiçbir boşluk bırakmaması
olağan değildir.

Ardından `mevcut_girdiler` ile `yasak_girdiler` listelerini karşılaştırınız. Ayrım
gerçekten yapılmışsa sistemin sızıntıya karşı ilk koruması kurulmuş demektir. Ayrım
yapılmamışsa istemi düzelterek kodu yeniden ürettiriniz.

Son olarak `oncelik` alanının kartta belirttiğiniz maliyet dengesine uyup uymadığına
bakınız. Araç kimi zaman kendi varsayılanına kayar ve kaçırmanın pahalı olduğu bir
problemde özgüllüğü önceler.


In [ ]:
for anahtar in ['oncelik', 'oncelik_gerekce']:
    print(f'{anahtar}: {spec.get(anahtar)}')
print()
print('Açık sorular:')
for soru in spec.get('acik_sorular', []):
    print(' -', soru)


## Bu defterde ele alınanlar

Problem yedi soruyla tanımlandı ve bu tanım yapılandırılmış bir belirtime dönüştürüldü.
Belirtim sonraki defterlerde kullanılacaktır.

Kart, üretken yapay zekâ aracına verilen ilk ve en belirleyici girdidir. Kartta yazmayan
hiçbir şey sonraki istemlerde dikkate alınmaz; araç eksik bilgiyi tahminle tamamlar ve
tahminini ayrıca bildirmez.
---

**Uyarı.** Bu defterde üretilen hiçbir çıktı doğrulanmış bir klinik araç değildir.
MIMIC-IV demo verisi tek bir Amerikan hastanesinden gelmektedir ve Türkiye'deki bir
yoğun bakım popülasyonunu temsil etmez. Materyal öğretim amaçlıdır.
